In [1]:
import joblib
import pandas as pd
import numpy as np

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report
)
import sys
sys.path.append('..')

from src.predict import predict_medical_plan
from src.feature_engineering import create_features

In [2]:
model = joblib.load(
    "../models/insurance_model.pkl"
)

1. Loading original cleaned dataset:

In [3]:
df = pd.read_csv(
    "../data/processed/clean_data.csv"
)

2. Loading feature eng. function that I created:

In [4]:
df = create_features(df)

3. Seperating target and feature variables:

In [5]:
X = df.drop(
    columns=["medical_plan"]
)

y = df["medical_plan"]

4. Final evaluation on test set:

In [6]:
X_test = joblib.load(
    "../models/X_test_xgb.pkl"
)

y_test = joblib.load(
    "../models/y_test_xgb.pkl"
)

In [7]:
preds = model.predict(X_test)

5. Predictiting final score:

In [8]:
print(
    classification_report(
        y_test,
        preds
    )
)

print(
    "Macro F1:",
    f1_score(
        y_test,
        preds,
        average="macro"
    )
)

              precision    recall  f1-score   support

           0       0.82      0.93      0.87        84
           1       0.72      0.74      0.73        31
           2       0.87      0.74      0.80        81

    accuracy                           0.82       196
   macro avg       0.80      0.80      0.80       196
weighted avg       0.82      0.82      0.82       196

Macro F1: 0.8005557033489993


6. Testing single customer prediction:

In [9]:
sample = pd.DataFrame({
    'user_id':[1001],
    'gender':['Male'],
    'age':[35],
    'state_tier':['Tier-1'],
    'occupation_class':['High-Risk'],
    'salary_bracket':['High'],
    'total_income_inr':[1200000],
    'is_smoker':[1], 
    'family_members':[4],
    'annual_expenditure_inr':[500000]
})

6.1 Applying FE:

In [10]:
sample = create_features(sample)

sample.head()

,user_id,gender,age,state_tier,occupation_class,salary_bracket,total_income_inr,is_smoker,family_members,annual_expenditure_inr,expense_ratio,savings,income_per_member,expenditure_per_member
0,1001,Male,35,Tier-1,High-Risk,High,1200000,1,4,500000,0.416667,700000,300000.0,125000.0


In [11]:
print(sample.columns)

Index(['user_id', 'gender', 'age', 'state_tier', 'occupation_class',
       'salary_bracket', 'total_income_inr', 'is_smoker', 'family_members',
       'annual_expenditure_inr', 'expense_ratio', 'savings',
       'income_per_member', 'expenditure_per_member'],
      dtype='str')


6.2 Predict:

In [12]:
prediction = model.predict(sample)

probabilities = model.predict_proba(sample)

prediction

c:\Users\kadit\MedicalPlan-Xray\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [3] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\kadit\MedicalPlan-Xray\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [3] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


array([0])

6.3 Probabilities used:

In [13]:
class_mapping = {
    0: "Low",
    1: "Medium",
    2: "High"
}

6.4 Making the mapped probabilities show actual string rather than encoded indexes:

In [14]:
pd.DataFrame(
    probabilities,
    columns=[
        class_mapping[i]
        for i in model.classes_
    ]
)

,Low,Medium,High
0,0.892683,0.01825,0.089067


In [15]:
joblib.dump(
    class_mapping,
    "../models/class_mapping.pkl"
)

['../models/class_mapping.pkl']

7. Testing predict.py is imported correctly or not via applying on sample data:

In [16]:
customer = {
    "user_id": 1001,
    "gender": "Male",
    "age": 32,
    "state_tier": "Tier-1",
    "occupation_class": "Professional",
    "salary_bracket": "50K-1L",
    "total_income_inr": 1200000,
    "is_smoker": 1,
    "family_members": 4,
    "annual_expenditure_inr": 500000
}

In [17]:
predict_medical_plan(customer)

c:\Users\kadit\MedicalPlan-Xray\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [2, 3] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\kadit\MedicalPlan-Xray\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [2, 3] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


{'prediction': 'Low',
 'probabilities': {'Low': 0.8927, 'Medium': 0.0183, 'High': 0.0891}}

8. Production Testing On Sample Users:

8.1 Customer 1:

In [18]:
customer_1 = {
    "user_id": 1,
    "gender": "Male",
    "age": 28,
    "state_tier": "Tier-1",
    "occupation_class": "Professional",
    "salary_bracket": "50K-1L",
    "total_income_inr": 1200000,
    "is_smoker": 1,
    "family_members": 4,
    "annual_expenditure_inr": 500000
}

In [19]:
predict_medical_plan(customer_1)

c:\Users\kadit\MedicalPlan-Xray\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [2, 3] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\kadit\MedicalPlan-Xray\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [2, 3] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


{'prediction': 'Low',
 'probabilities': {'Low': 0.892, 'Medium': 0.019, 'High': 0.089}}

8.2 Customer 2:

In [20]:
customer_2 = {
    "user_id": 2,
    "gender": "Female",
    "age": 35,
    "state_tier": "Tier-2",
    "occupation_class": "Service",
    "salary_bracket": "25K-50K",
    "total_income_inr": 600000,
    "is_smoker": 0,
    "family_members": 5,
    "annual_expenditure_inr": 300000
}

In [21]:
predict_medical_plan(customer_2)

c:\Users\kadit\MedicalPlan-Xray\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [2, 3] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\kadit\MedicalPlan-Xray\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [2, 3] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


{'prediction': 'Low',
 'probabilities': {'Low': 0.8659, 'Medium': 0.021, 'High': 0.1131}}

8.3 Customer 3:

In [22]:
customer_3 = {
    "user_id": 3,
    "gender": "Male",
    "age": 22,
    "state_tier": "Tier-3",
    "occupation_class": "Student",
    "salary_bracket": "0-25K",
    "total_income_inr": 200000,
    "is_smoker": 0,
    "family_members": 2,
    "annual_expenditure_inr": 120000
}

8.4 Customer 4:

In [23]:
customer_4 = {
    "user_id": 4,
    "gender": "Female",
    "age": 50,
    "state_tier": "Tier-1",
    "occupation_class": "Business",
    "salary_bracket": "1L+",
    "total_income_inr": 2500000,
    "is_smoker": 0,
    "family_members": 6,
    "annual_expenditure_inr": 900000
}

8.5 Customer 5:

In [24]:
customer_5 = {
    "user_id": 5,
    "gender": "Male",
    "age": 65,
    "state_tier": "Tier-2",
    "occupation_class": "Retired",
    "salary_bracket": "25K-50K",
    "total_income_inr": 500000,
    "is_smoker": 1,
    "family_members": 1,
    "annual_expenditure_inr": 450000
}

In [25]:
customers = [
    customer_1,
    customer_2,
    customer_3,
    customer_4,
    customer_5
]

for i, customer in enumerate(customers, 1):
    print(f"\nCustomer {i}")
    print(predict_medical_plan(customer))

c:\Users\kadit\MedicalPlan-Xray\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [2, 3] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\kadit\MedicalPlan-Xray\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [2, 3] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\kadit\MedicalPlan-Xray\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [2, 3] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\kadit\MedicalPlan-Xray\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [2, 3] during transform. These unknown categories will be encoded as all zeros
  warn


Customer 1
{'prediction': 'Low', 'probabilities': {'Low': 0.892, 'Medium': 0.019, 'High': 0.089}}

Customer 2
{'prediction': 'Low', 'probabilities': {'Low': 0.8659, 'Medium': 0.021, 'High': 0.1131}}

Customer 3
{'prediction': 'Medium', 'probabilities': {'Low': 0.0339, 'Medium': 0.7283, 'High': 0.2378}}

Customer 4
{'prediction': 'Low', 'probabilities': {'Low': 0.9149, 'Medium': 0.0119, 'High': 0.0732}}

Customer 5
{'prediction': 'Low', 'probabilities': {'Low': 0.8474, 'Medium': 0.056, 'High': 0.0966}}


c:\Users\kadit\MedicalPlan-Xray\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [2, 3] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
c:\Users\kadit\MedicalPlan-Xray\venv\Lib\site-packages\sklearn\preprocessing\_encoders.py:261: UserWarning: Found unknown categories in columns [2, 3] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


# Conclusion : Pipeline is wokring correctly!!!

In [26]:
high_customer = (
    df[df['medical_plan']=="High"]
    .iloc[0]
    .drop('medical_plan')
    .to_dict()
)

high_customer

{'user_id': 3,
 'gender': 'Male',
 'age': 46,
 'state_tier': 'Tier-2',
 'occupation_class': 'High-Risk',
 'salary_bracket': 'Tier-1',
 'total_income_inr': 233707,
 'is_smoker': 0,
 'family_members': 3,
 'annual_expenditure_inr': 128494,
 'expense_ratio': 0.5498080930395752,
 'savings': 105213,
 'income_per_member': 77902.33333333333,
 'expenditure_per_member': 42831.333333333336}

In [27]:
predict_medical_plan(high_customer)

{'prediction': 'High',
 'probabilities': {'Low': 0.1185, 'Medium': 0.1307, 'High': 0.7508}}

In [28]:
print(df[['total_income_inr',
          'annual_expenditure_inr',
          'family_members',
          'age']].describe())

       total_income_inr  annual_expenditure_inr  family_members         age
count      9.800000e+02              980.000000      980.000000  980.000000
mean       9.241866e+05           134662.093878        3.527551   45.778571
std        8.652394e+05            65517.903020        1.709869   16.279509
min        1.519340e+05             5000.000000        1.000000   18.000000
25%        4.117075e+05            84310.000000        2.000000   32.000000
50%        7.894210e+05           124310.000000        3.500000   45.000000
75%        1.124310e+06           184096.750000        5.000000   60.000000
max        4.997509e+06           416505.000000        6.000000   75.000000


In [29]:
print(df['total_income_inr'].max())
print(df['annual_expenditure_inr'].max())
print(df['family_members'].max())
print(df['age'].max())

4997509
416505
6
75
